# Feature engineering — ESA anomaly benchmark

Builds the feature tables, labels, and chronological splits that the anomaly-detection model will be trained on, using **only the selected real ESA telemetry**.

**Inputs**

Selection rationale and the full channel screening are in `data_exploration.ipynb` (Part C).

| Role | Source |
| --- | --- |
| Primary benchmark | Mission 1 group 8 — `channel_41` … `channel_46` (15,381,169 samples each, 30 s cadence, 2000–2013) |
| Simple baseline stream | Mission 1 `channel_66` (4,810,995 samples, ~158 s cadence) |
| Secondary comparison streams | Mission 2 `channel_13`, `channel_20` (7,387,262 samples each, 18 s cadence, 2000–2003) |

**Outputs** (written under `/content/feature_engineering/`)

| Artifact | Contents |
| --- | --- |
| `features/<mission>/<channel>_windows.parquet` | per-channel window features, labels, split |
| `features/Mission1/group8_windows.parquet` | joined group-8 table with cross-channel features |
| `features/labels_windows.parquet` | long-format label table (one row per window × category) |
| `features/train_stats.json` | train-only median/MAD/percentiles per feature, for scaling |
| `features/manifest.json` | schema, cadence, splits, feature list, row and label counts |
| `feature_tables.zip` | everything above, ready to download |

**Scope rules**

- Real ESA telemetry only. No generated/mock values and no Raspberry Pi readings enter these features.
- Channels stay anonymized. Nothing here claims `channel_41` is a named physical sensor.
- Labels keep `Anomaly`, `Rare Event`, and `Communication Gap` separate. Only `Anomaly` is the default positive class.
- Native cadence is preserved. Windows are wall-clock based, never resampled to an artificial rate.

**Runtime.** Expect ~10–25 minutes on a Colab CPU runtime: ~600 MiB of channel data plus windowing of ~60 M samples. Set `QUICK_MODE = True` below to validate the notebook on one channel and a short period first.


## 1. Feature design

### Why windows instead of raw samples

A 30 s stream over 13 years is ~15 M rows per channel. Point-level features at that size are memory-prohibitive and do not align with the benchmark's label semantics, which are **intervals**, not single samples. So the primary artifact is a window table:

- **Window length: 10 minutes** → ~20 samples per window at 30 s cadence, ~683 k windows per channel over 13 years.
- Windows are aligned to wall-clock time (`floor(timestamp, 10 min)`), so channels with different cadences produce comparable timestamps.
- `n_samples` and `coverage` record how much data a window actually contains, so sparse windows can be filtered rather than silently interpolated.

### Feature groups

| Group | Examples | Purpose |
| --- | --- | --- |
| Level | `v_mean`, `v_median`, `v_q05`, `v_q95` | Where the signal sits |
| Dispersion | `v_std`, `v_range`, `v_iqr` | How much it moves within the window |
| Extremes | `v_min`, `v_max`, `below_train_band`, `above_train_band` | Spikes and saturation |
| Dynamics | `abs_delta_sum`, `abs_delta_max`, `rate_max`, `slope_long` | Rate of change — the feature that separated labelled from nominal data during screening |
| Shape / state | `zero_frac`, `is_zero_window`, `is_flat`, `is_gappy` | Stuck-at-zero, flatline, dropouts |
| Data quality | `n_samples`, `coverage`, `gap_frac`, `sample_delta` | Distinguishes a real change from missing data |
| Context (causal) | `mean_delta_w`, `roll_med_ctx`, `robust_z_ctx`, `roll_std_long` | Compares the window against its own recent past |
| Cross-channel (group 8 only) | `group_v_mean`, `group_v_std`, `pairdiff_*`, `group_max_abs_z` | Exploits the three correlated channel pairs |

### Leakage rules

1. Every context feature is **causal**: rolling statistics look backwards only (`rolling`, `shift`), never centred.
2. Any fitted quantity — train band bounds, feature medians/MADs — is computed from **train-split nominal windows only** and stored in `train_stats.json`.
3. Splits are chronological and assigned by window start, **per mission**. Mission 2's archive ends in mid-2003, so it needs its own boundaries: a Mission-1-style split would put every Mission 2 window in `train` and leave nothing to evaluate.

   | Mission | train | val | test |
   | --- | --- | --- | --- |
   | Mission 1 (group 8, `channel_66`) | 2000-01-01 → 2010-01-01 | 2010-01-01 → 2012-01-01 | 2012-01-01 → 2014-01-01 |
   | Mission 2 (`channel_13`, `channel_20`) | 2000-01-01 → 2001-06-01 | 2001-06-01 → 2002-06-01 | 2002-06-01 → 2004-01-01 |

4. Windows whose label interval crosses a split boundary are flagged `label_crosses_split` and excluded from evaluation.
5. `Rare Event` and `Communication Gap` are never folded into the positive class.


In [1]:
%pip install -q zipfile-deflate64 remotezip

from pathlib import Path
import io
import json
import zipfile

import numpy as np
import pandas as pd

import zipfile_deflate64  # noqa: F401  patches zipfile for Deflate64 entries
from remotezip import RemoteZip

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

# ------------------------------------------------------------------ paths
WORK_DIR = Path("/content/feature_engineering")
CACHE_DIR = Path("/content/cache")          # raw channel ZIPs, reused across runs
FEATURE_DIR = WORK_DIR / "features"
for _path in (WORK_DIR, CACHE_DIR, FEATURE_DIR):
    _path.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- sources
MISSIONS = {
    "Mission1": "https://zenodo.org/records/15237121/files/ESA-Mission1.zip",
    "Mission2": "https://zenodo.org/records/15237121/files/ESA-Mission2.zip",
}

# ------------------------------------------------------- selected data
PRIMARY_GROUP = ("Mission1", 8)
GROUP8_CHANNELS = [(PRIMARY_GROUP[0], f"channel_{i}") for i in range(41, 47)]

CHANNELS = GROUP8_CHANNELS + [
    ("Mission2", "channel_13"),   # secondary stream A
    ("Mission2", "channel_20"),   # secondary stream B
    ("Mission1", "channel_66"),   # simple baseline stream
]

# --------------------------------------------------------------- shaping
WINDOW_MINUTES = 10
CONTEXT_WINDOWS = 6            # 6 windows = 1 hour
LONG_CONTEXT_WINDOWS = 12      # 12 windows = 2 hours

# Mission 2 ends in mid-2003, so it needs its own boundaries; a shared Mission-1
# profile would put every Mission 2 window in `train` and leave nothing to evaluate.
SPLIT_PROFILES = {
    "Mission1": {
        "train": ("2000-01-01", "2010-01-01"),
        "val":   ("2010-01-01", "2012-01-01"),
        "test":  ("2012-01-01", "2014-01-01"),
    },
    "Mission2": {
        "train": ("2000-01-01", "2001-06-01"),
        "val":   ("2001-06-01", "2002-06-01"),
        "test":  ("2002-06-01", "2004-01-01"),
    },
}

LABEL_CATEGORIES = ["Anomaly", "Rare Event", "Communication Gap"]
POSITIVE_CATEGORY = "Anomaly"

MAX_CHANNEL_ZIP_MIB = 200
SCHEMA_VERSION = "1.0"

# Set True for a fast end-to-end check: one channel, last 18 months only.
QUICK_MODE = False
QUICK_CHANNEL = ("Mission1", "channel_66")
QUICK_DAYS = 550

print(f"window={WINDOW_MINUTES} min | channels={len(CHANNELS)} | schema={SCHEMA_VERSION}")


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
window=10 min | channels=9 | schema=1.0


## 2. Channel access and caching

`RemoteZip` reads the archive's central directory and only the requested nested entry, so the full 3.5–3.9 GiB archive is never downloaded. Each channel arrives as its own nested ZIP containing one pickled DataFrame (`index` = timestamps, columns = channel value + `label`).

The first run caches each nested ZIP under `/content/cache/`, which makes re-runs fast and survives transient Zenodo failures. Delete that folder to force a fresh download.

> Channel sizes: group 8 ≈ 80 MiB each, `channel_66` ≈ 22 MiB, Mission 2 ≈ 38–47 MiB. Total ≈ 600 MiB on the first run.


In [2]:
def _cached_channel_zip(mission, channel):
    return CACHE_DIR / mission / f"{channel}.zip"


def fetch_channel_zip(mission, channel, refresh=False):
    """Return the bytes of one nested channel ZIP, downloading only if uncached."""
    path = _cached_channel_zip(mission, channel)
    if path.exists() and not refresh:
        return path.read_bytes(), "cache"

    suffix = f"/channels/{channel}.zip"
    last_error = None
    for _attempt in range(3):
        try:
            with RemoteZip(MISSIONS[mission], timeout=120) as outer:
                matches = [n for n in outer.namelist() if n.endswith(suffix)]
                if not matches:
                    raise FileNotFoundError(f"{mission}/{channel}.zip not found")
                member = matches[0]
                size_mib = outer.getinfo(member).file_size / 1024 ** 2
                if size_mib > MAX_CHANNEL_ZIP_MIB:
                    raise ValueError(
                        f"{mission}/{channel} is {size_mib:.0f} MiB, "
                        f"above the {MAX_CHANNEL_ZIP_MIB} MiB limit"
                    )
                payload = outer.read(member)
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(payload)
            return payload, f"downloaded {size_mib:.0f} MiB"
        except Exception as exc:  # noqa: BLE001 - Colab networks drop ranges often
            last_error = exc
    raise RuntimeError(f"Could not fetch {mission}/{channel}") from last_error


def load_channel_frame(mission, channel, refresh=False):
    payload, origin = fetch_channel_zip(mission, channel, refresh=refresh)
    with zipfile.ZipFile(io.BytesIO(payload)) as nested:
        files = [n for n in nested.namelist() if not n.endswith("/")]
        if len(files) != 1:
            raise ValueError(f"Expected one data file in {channel}, found {files}")
        with nested.open(files[0]) as handle:
            frame = pd.read_pickle(handle)
    print(f"{mission}/{channel}: {len(frame):,} rows ({origin})")
    return frame


def to_series(frame, channel):
    """Clean, sorted, UTC-indexed numeric Series for one channel."""
    value_columns = [c for c in frame.columns if c != "label"]
    column = channel if channel in value_columns else value_columns[0]
    index = pd.DatetimeIndex(pd.to_datetime(frame.index, errors="coerce", utc=True))
    values = pd.to_numeric(frame[column], errors="coerce").to_numpy(dtype="float64")
    series = pd.Series(values, index=index, name=channel).dropna().sort_index()
    return series[~series.index.duplicated(keep="first")]


# Smoke test the reader on the smallest selected channel.
if QUICK_MODE:
    _probe = to_series(load_channel_frame(*QUICK_CHANNEL), QUICK_CHANNEL[1])
    print(_probe.head(3))
    print("span:", _probe.index.min(), "->", _probe.index.max())


## 3. Labels

`labels.csv` gives `(Channel, StartTime, EndTime, ID)`; `anomaly_types.csv` maps each `ID` to `Category`, `Class`, and `Subclass`.

Two properties of this dataset shape the label handling:

- **Intervals repeat across channels.** In group 8 the same event appears once per channel, so per-channel counts (≈52 anomalies each) are *not* 6× as many events. The group-level union is 98 `Anomaly`, 294 `Rare Event`, 4 `Communication Gap`.
- **Zero-duration annotations exist** (e.g. the 2007-04-15 event). These are treated as marking the containing window, with an overlap fraction of one sample rather than a full window.

Labels are attached as `label_<category>` (boolean overlap) plus `label_<category>_frac` (fraction of the window covered), so the model can choose hard or graded targets.


In [3]:
def load_mission_labels(mission):
    """Read and cache labels.csv + anomaly_types.csv for one mission."""
    cache = CACHE_DIR / mission / "labels.parquet"
    if cache.exists():
        return pd.read_parquet(cache)

    with RemoteZip(MISSIONS[mission], timeout=120) as outer:
        names = outer.namelist()
        labels_name = next(n for n in names if n.lower().endswith("/labels.csv"))
        types_name = next(n for n in names if n.lower().endswith("/anomaly_types.csv"))
        with outer.open(labels_name) as handle:
            labels = pd.read_csv(handle)
        with outer.open(types_name) as handle:
            types = pd.read_csv(handle)

    labels = labels.merge(
        types[["ID", "Category", "Class", "Subclass"]], on="ID", how="left"
    )
    labels["Mission"] = mission
    for column in ("StartTime", "EndTime"):
        labels[column] = pd.to_datetime(labels[column], errors="coerce", utc=True)
    labels = labels.dropna(subset=["StartTime", "EndTime"])

    cache.parent.mkdir(parents=True, exist_ok=True)
    labels.to_parquet(cache, index=False)
    return labels


label_frames = [load_mission_labels(m) for m in sorted({m for m, _ in CHANNELS})]
labels_all = pd.concat(label_frames, ignore_index=True)
labels_all["Category"] = labels_all["Category"].astype("string").str.strip()

print("label rows per mission/category:")
display(
    labels_all.groupby(["Mission", "Category"]).size().unstack(fill_value=0)
)

print("labels available for the selected channels:")
display(
    labels_all[labels_all.apply(lambda r: (r["Mission"], r["Channel"]) in set(CHANNELS), axis=1)]
    .groupby(["Mission", "Channel", "Category"]).size().unstack(fill_value=0)
)


label rows per mission/category:


Category,Anomaly,Communication Gap,Rare Event
Mission,,,
Mission1,1203,208,2178
Mission2,186,0,10981


labels available for the selected channels:


Category             Anomaly  Communication Gap  Rare Event
Mission  Channel                                           
Mission1 channel_41       54                  4         154
         channel_42       53                  4         154
         channel_43       53                  4         154
         channel_44       55                  4         151
         channel_45       53                  4         151
         channel_46       52                  4         147
         channel_66       10                  0           6
Mission2 channel_13       11                  0         621
         channel_20       12                  0         620

## 4. Window features

`build_channel_windows` does four things in one pass:

1. **Cadence estimate** — median positive gap, used to separate real movement from outages.
2. **Point-level deltas** — `Δvalue`, `|Δvalue|`, and rate per minute, masked when the preceding gap exceeds 3× the native cadence. This prevents a 20-day outage from registering as a huge jump.
3. **Window aggregation** — group by 10-minute wall-clock bin, then compute level, dispersion, extremes, dynamics, and quality statistics.
4. **Causal context** — reindex onto a complete 10-minute grid (so gaps are explicit, not skipped), then compute backward-looking deviation features: `roll_med_ctx`, `roll_mad_ctx`, `robust_z_ctx`, `roll_std_long`, `slope_long`.

`coverage` and `n_samples` matter: a window with 2 samples out of an expected 20 is not evidence of anything, and the model dev can drop or flag it.


In [4]:
def window_grid(start, end, minutes):
    """Complete, gap-explicit wall-clock grid at the window cadence."""
    start = pd.Timestamp(start).tz_convert("UTC") if pd.Timestamp(start).tz else pd.Timestamp(start, tz="UTC")
    end = pd.Timestamp(end).tz_convert("UTC") if pd.Timestamp(end).tz else pd.Timestamp(end, tz="UTC")
    return pd.date_range(
        start=start.floor(f"{minutes}min"),
        end=end.ceil(f"{minutes}min"),
        freq=f"{minutes}min",
        inclusive="left",
    )


def _slope(values, span_windows, minutes):
    """Secant slope over the last `span_windows`, per hour. Causal and cheap."""
    hours = (span_windows - 1) * minutes / 60.0
    return (values - values.shift(span_windows - 1)) / hours


def build_channel_windows(series, window_minutes=WINDOW_MINUTES):
    """Aggregate one channel to fixed wall-clock windows plus causal context."""
    series = series.sort_index()
    if len(series) < 20:
        raise ValueError(f"Not enough samples for {series.name}")

    times = series.index
    gap_s = pd.Series(np.diff(times.asi8) / 1e9, index=times[1:]).reindex(times)
    positive_gaps = gap_s.to_numpy()[1:]
    cadence_s = float(np.nanmedian(positive_gaps[positive_gaps > 0]))
    usable = gap_s.gt(0) & gap_s.le(3 * cadence_s)

    value = series.astype("float64")
    delta = value.diff().where(usable)
    frame = pd.DataFrame(
        {
            "win": times.floor(f"{window_minutes}min").to_numpy(),
            "value": value.to_numpy(),
            "abs_delta": delta.abs().to_numpy(),
            "rate": (delta / (gap_s / 60.0)).where(usable).to_numpy(),
            "is_zero": value.eq(0).to_numpy(),
            "is_gap": (~usable).fillna(False).to_numpy(),
        }
    )

    grouped = frame.groupby("win", sort=True)
    windows = pd.DataFrame(
        {
            "n_samples": grouped["value"].size(),
            "v_mean": grouped["value"].mean(),
            "v_std": grouped["value"].std(),
            "v_min": grouped["value"].min(),
            "v_max": grouped["value"].max(),
            "v_median": grouped["value"].median(),
            "v_q05": grouped["value"].quantile(0.05),
            "v_q95": grouped["value"].quantile(0.95),
            "v_first": grouped["value"].first(),
            "v_last": grouped["value"].last(),
            "abs_delta_sum": grouped["abs_delta"].sum(),
            "abs_delta_max": grouped["abs_delta"].max(),
            "rate_max": grouped["rate"].max(),
            "zero_frac": grouped["is_zero"].mean(),
            "gap_frac": grouped["is_gap"].mean(),
        }
    )
    windows["v_range"] = windows["v_max"] - windows["v_min"]
    windows["v_iqr"] = windows["v_q95"] - windows["v_q05"]

    expected_samples = max((window_minutes * 60.0) / cadence_s, 1.0)
    grid = window_grid(times.min(), times.max(), window_minutes)
    windows = windows.reindex(grid)
    windows["n_samples"] = windows["n_samples"].fillna(0.0)
    windows["coverage"] = (windows["n_samples"] / expected_samples).clip(upper=1.0)
    windows["is_present"] = windows["n_samples"].gt(0)

    level = windows["v_mean"]
    ctx, long_ctx = CONTEXT_WINDOWS, LONG_CONTEXT_WINDOWS
    windows["mean_delta_w"] = level.diff()
    windows["roll_med_ctx"] = level.rolling(
        ctx, min_periods=max(2, ctx // 2)
    ).median()
    deviation = level - windows["roll_med_ctx"]
    windows["roll_mad_ctx"] = deviation.abs().rolling(
        ctx, min_periods=max(2, ctx // 2)
    ).median()
    windows["dev_med_ctx"] = deviation
    windows["robust_z_ctx"] = deviation / (1.4826 * windows["roll_mad_ctx"] + 1e-9)
    windows["roll_std_long"] = level.rolling(
        long_ctx, min_periods=max(3, long_ctx // 3)
    ).std()
    windows["slope_long"] = _slope(level, long_ctx, window_minutes)

    windows["is_zero_window"] = windows["zero_frac"].gt(0).fillna(False)
    windows["is_flat"] = (
        windows["abs_delta_sum"].fillna(0).eq(0) & windows["n_samples"].ge(3)
    )
    windows["is_gappy"] = windows["coverage"].lt(0.5)

    windows.index.name = "window_start"
    return windows, {"cadence_s": cadence_s, "expected_samples": expected_samples}


# Sanity check on a synthetic series before touching real data.
_probe_index = pd.date_range("2001-01-01", periods=6000, freq="30s", tz="UTC")
_probe = pd.Series(np.arange(6000, dtype="float64") % 97, index=_probe_index, name="probe")
_probe_windows, _probe_meta = build_channel_windows(_probe)
print("probe windows:", len(_probe_windows), "| cadence:", _probe_meta)
display(_probe_windows.head(3))


probe windows: 300 | cadence: {'cadence_s': 30.0, 'expected_samples': 20.0}


,n_samples,v_mean,v_std,v_min,v_max,v_median,v_q05,v_q95,v_first,v_last,abs_delta_sum,abs_delta_max,rate_max,zero_frac,gap_frac,v_range,v_iqr,coverage,is_present,mean_delta_w,roll_med_ctx,roll_mad_ctx,dev_med_ctx,robust_z_ctx,roll_std_long,slope_long,is_zero_window,is_flat,is_gappy
window_start,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2001-01-01 00:00:00+00:00,20,9.5,5.91608,0.0,19.0,9.5,0.95,18.05,0.0,19.0,19.0,1.0,2.0,0.05,0.05,19.0,17.1,1.0,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,False,False
2001-01-01 00:10:00+00:00,20,29.5,5.91608,20.0,39.0,29.5,20.95,38.05,20.0,39.0,20.0,1.0,2.0,0.00,0.00,19.0,17.1,1.0,True,20.0,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False
2001-01-01 00:20:00+00:00,20,49.5,5.91608,40.0,59.0,49.5,40.95,58.05,40.0,59.0,20.0,1.0,2.0,0.00,0.00,19.0,17.1,1.0,True,20.0,29.5,NaN,20.0,NaN,NaN,NaN,False,False,False


## 5. Aligning labels to windows

Each annotation interval is mapped onto the window grid it overlaps. Two derived columns per category:

- `label_<category>` — the window overlaps the interval at all.
- `label_<category>_frac` — fraction of the window covered, useful for graded targets and for excluding edge windows.

A `label_crosses_split` flag marks windows whose annotation interval spans a split boundary. Those are excluded from evaluation to avoid training on a fault that is half in test.


In [5]:
def merge_intervals(intervals):
    """Merge overlapping/adjacent (start, end) pairs."""
    merged = []
    for start, end in sorted(intervals):
        if end < start:
            start, end = end, start
        if merged and start <= merged[-1][1]:
            previous_start, previous_end = merged[-1]
            merged[-1] = (previous_start, max(previous_end, end))
        else:
            merged.append((start, end))
    return merged


def channel_intervals(labels, mission, channel, category):
    subset = labels[
        labels["Mission"].eq(mission)
        & labels["Channel"].eq(channel)
        & labels["Category"].eq(category)
    ]
    if subset.empty:
        return []
    pairs = list(subset[["StartTime", "EndTime"]].drop_duplicates().itertuples(index=False))
    return merge_intervals([(s, e) for s, e in pairs])


def attach_labels(windows, intervals_by_category, mission, window_minutes=WINDOW_MINUTES):
    """Attach boolean + fractional overlap labels to a regular window table."""
    windows = windows.copy()
    grid = windows.index
    origin = int(grid[0].value)
    step = int(pd.Timedelta(minutes=window_minutes).value)
    count = len(grid)

    for category, intervals in intervals_by_category.items():
        slug = category.lower().replace(" ", "_")
        flag = np.zeros(count, dtype=bool)
        frac = np.zeros(count, dtype="float32")
        crosses = np.zeros(count, dtype=bool)

        for start, end in intervals:
            first = max(int(np.floor((start.value - origin) / step)), 0)
            last = min(int(np.floor((end.value - origin) / step)), count - 1)
            if last < first:
                continue
            flag[first : last + 1] = True
            frac[first : last + 1] = 1.0

            if end.value == start.value:          # zero-duration annotation
                samples = max(int(windows["n_samples"].iloc[first]), 1)
                frac[first] = min(1.0, 1.0 / samples)
            elif first == last:                    # annotation entirely inside one window
                frac[first] = min(1.0, (end.value - start.value) / step)
            else:
                window_start = origin + first * step
                if start.value > window_start:
                    frac[first] = min(1.0, (window_start + step - start.value) / step)
                window_start = origin + last * step
                if end.value < window_start + step:
                    frac[last] = min(1.0, (end.value - window_start) / step)

            split_first = assign_split(grid[[first]], mission).iloc[0]
            split_last = assign_split(grid[[last]], mission).iloc[0]
            if split_first != split_last:
                crosses[first : last + 1] = True

        windows[f"label_{slug}"] = flag
        windows[f"label_{slug}_frac"] = frac
        if crosses.any():
            windows["label_crosses_split"] = (
                windows.get("label_crosses_split", False) | pd.Series(crosses, index=grid)
            )

    if "label_crosses_split" not in windows:
        windows["label_crosses_split"] = False
    return windows


def assign_split(index, mission):
    """Chronological split label for a window index, using that mission's profile."""
    if mission not in SPLIT_PROFILES:
        raise KeyError(f"No split profile configured for {mission}")
    out = pd.Series("unassigned", index=index, dtype="string")
    for name, (start, end) in SPLIT_PROFILES[mission].items():
        mask = (index >= pd.Timestamp(start, tz="UTC")) & (index < pd.Timestamp(end, tz="UTC"))
        out[mask] = name
    return out


print("helpers ready")


helpers ready


## 6. Build the per-channel feature tables

One channel at a time, so peak memory stays around one window table (~700 k rows) plus its intermediates:

1. read → sort → drop duplicate timestamps,
2. build windows and causal context,
3. attach the three label categories,
4. assign the chronological split,
5. add train-only band flags and compute train statistics,
6. write Parquet, run the baseline reference, keep only the group-relevant columns in memory.

`train_stats.json` is deliberately computed from **train-split nominal windows only** (no `Anomaly`, no `Rare Event`). Use it to standardise features in the model repo; do not refit on validation or test.


In [6]:
GROUP_KEEP = [
    "v_mean", "v_std", "abs_delta_sum", "robust_z_ctx",
    "zero_frac", "slope_long", "coverage", "n_samples",
]

PAIR_CANDIDATES = [
    ("channel_41", "channel_44"),
    ("channel_42", "channel_45"),
    ("channel_43", "channel_46"),
]


def numeric_feature_columns(table):
    return [
        column
        for column in table.columns
        if column not in ("split", "is_present")
        and not column.startswith("label_")
        and pd.api.types.is_numeric_dtype(table[column])
    ]


def add_train_bounds(windows):
    """Train-only nominal band flags; returns the bounds that were used."""
    train_nominal = (
        windows["split"].eq("train")
        & ~windows["label_anomaly"]
        & ~windows["label_rare_event"]
        & windows["v_mean"].notna()
    )
    level = windows.loc[train_nominal, "v_mean"]
    low, high = float(level.quantile(0.001)), float(level.quantile(0.999))
    windows["below_train_band"] = windows["v_mean"].lt(low)
    windows["above_train_band"] = windows["v_mean"].gt(high)
    return {"train_v_mean_p001": low, "train_v_mean_p999": high}


def compute_train_stats(windows, features):
    train = windows["split"].eq("train")
    stats = {}
    for feature in features:
        values = windows.loc[train, feature].dropna()
        if values.empty:
            continue
        median = float(values.median())
        stats[feature] = {
            "median": median,
            "mad": float((values - median).abs().median()),
            "p01": float(values.quantile(0.01)),
            "p99": float(values.quantile(0.99)),
        }
    return stats


def build_one_channel(mission, channel):
    frame = load_channel_frame(mission, channel)
    series = to_series(frame, channel)
    del frame

    if QUICK_MODE:
        cutoff = series.index.max() - pd.Timedelta(days=QUICK_DAYS)
        series = series[series.index >= cutoff]
        print(f"  quick mode: {len(series):,} samples since {cutoff.date()}")

    windows, meta = build_channel_windows(series)
    intervals = {
        category: channel_intervals(labels_all, mission, channel, category)
        for category in LABEL_CATEGORIES
    }
    windows = attach_labels(windows, intervals, mission)
    windows["split"] = assign_split(windows.index, mission)
    bounds = add_train_bounds(windows)

    # Boolean flags become 0/1 so they are valid numeric features and Parquet-safe.
    for column in windows.select_dtypes(include="bool").columns:
        windows[column] = windows[column].astype("int8")

    features = numeric_feature_columns(windows)
    stats = compute_train_stats(windows, features)

    out_path = FEATURE_DIR / mission / f"{channel}_windows.parquet"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    export = windows.copy()
    for column in export.select_dtypes("float64").columns:
        export[column] = export[column].astype("float32")
    export.to_parquet(out_path, compression="snappy")

    summary = {
        "Mission": mission,
        "Channel": channel,
        "rows": len(windows),
        "samples": len(series),
        "cadence_s": meta["cadence_s"],
        "window_minutes": WINDOW_MINUTES,
        "start": str(windows.index.min()),
        "end": str(windows.index.max()),
        "anomaly_windows": int(windows["label_anomaly"].sum()),
        "rare_event_windows": int(windows["label_rare_event"].sum()),
        "communication_gap_windows": int(windows["label_communication_gap"].sum()),
        "train_rows": int(windows["split"].eq("train").sum()),
        "val_rows": int(windows["split"].eq("val").sum()),
        "test_rows": int(windows["split"].eq("test").sum()),
        "path": str(out_path),
        "features": features,
        "train_bounds": bounds,
        "train_stats": stats,
    }

    # Keep only what the group table needs, to bound memory.
    keep = [c for c in windows.columns if c in GROUP_KEEP or c.startswith("label_") or c == "split"]
    reduced = windows[keep].copy()
    for column in reduced.select_dtypes("float64").columns:
        reduced[column] = reduced[column].astype("float32")

    print(
        f"  -> {len(windows):,} windows | "
        f"anomaly={summary['anomaly_windows']} "
        f"rare={summary['rare_event_windows']} | "
        f"{out_path.name}"
    )
    return windows, reduced, summary


selected = [QUICK_CHANNEL] if QUICK_MODE else CHANNELS
summaries = []
full_windows = {}
group_inputs = {}

for mission, channel in selected:
    print(f"\n=== {mission}/{channel} ===")
    windows, reduced, summary = build_one_channel(mission, channel)
    summaries.append(summary)
    if (mission, channel) in GROUP8_CHANNELS:
        group_inputs[channel] = reduced
    del windows  # keep only the reduced copy for group-8 channels

summary_table = pd.DataFrame(summaries).drop(
    columns=["train_stats", "train_bounds", "features"]
)
display(summary_table)



=== Mission1/channel_41 ===
Mission1/channel_41: 15,381,169 rows (downloaded 79 MiB)
  -> 736,416 windows | anomaly=5567 rare=5735 | channel_41_windows.parquet

=== Mission1/channel_42 ===
Mission1/channel_42: 15,381,169 rows (downloaded 82 MiB)
  -> 736,416 windows | anomaly=5570 rare=5740 | channel_42_windows.parquet

=== Mission1/channel_43 ===
Mission1/channel_43: 15,381,169 rows (downloaded 79 MiB)
  -> 736,416 windows | anomaly=5596 rare=5741 | channel_43_windows.parquet

=== Mission1/channel_44 ===
Mission1/channel_44: 15,381,169 rows (downloaded 81 MiB)
  -> 736,416 windows | anomaly=5592 rare=5740 | channel_44_windows.parquet

=== Mission1/channel_45 ===
Mission1/channel_45: 15,381,169 rows (downloaded 80 MiB)
  -> 736,416 windows | anomaly=5562 rare=5740 | channel_45_windows.parquet

=== Mission1/channel_46 ===
Mission1/channel_46: 15,381,169 rows (downloaded 82 MiB)
  -> 736,416 windows | anomaly=5574 rare=5725 | channel_46_windows.parquet

=== Mission2/channel_13 ===
Missi

,Mission,Channel,rows,samples,cadence_s,window_minutes,start,end,anomaly_windows,rare_event_windows,communication_gap_windows,train_rows,val_rows,test_rows,path
0,Mission1,channel_41,736416,15381169,30.0,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,5567,5735,242,526032,105120,105264,/content/feature_engineering/features/Mission1...
1,Mission1,channel_42,736416,15381169,30.0,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,5570,5740,242,526032,105120,105264,/content/feature_engineering/features/Mission1...
2,Mission1,channel_43,736416,15381169,30.0,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,5596,5741,242,526032,105120,105264,/content/feature_engineering/features/Mission1...
3,Mission1,channel_44,736416,15381169,30.0,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,5592,5740,242,526032,105120,105264,/content/feature_engineering/features/Mission1...
4,Mission1,channel_45,736416,15381169,30.0,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,5562,5740,242,526032,105120,105264,/content/feature_engineering/features/Mission1...
5,Mission1,channel_46,736416,15381169,30.0,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,5574,5725,242,526032,105120,105264,/content/feature_engineering/features/Mission1...
6,Mission2,channel_13,183888,7387262,18.0,10,2000-01-01 00:00:00+00:00,2003-06-30 23:50:00+00:00,66,4967,0,74448,52560,56880,/content/feature_engineering/features/Mission2...
7,Mission2,channel_20,183888,7387262,18.0,10,2000-01-01 00:00:00+00:00,2003-06-30 23:50:00+00:00,46,5307,0,74448,52560,56880,/content/feature_engineering/features/Mission2...
8,Mission1,channel_66,736416,4810995,157.5,10,2000-01-01 00:00:00+00:00,2013-12-31 23:50:00+00:00,7826,1237,0,526032,105120,105264,/content/feature_engineering/features/Mission1...


## 7. Group-level features (group 8)

Screening found three correlated pairs inside group 8 (`41/44` ρ≈0.92, `42/45` ρ≈0.90, `43/46` ρ≈0.89). That structure is the main reason group 8 is the primary benchmark, so the joined table exposes it directly:

| Feature | Meaning |
| --- | --- |
| `group_v_mean`, `group_v_std` | Common level and disagreement across the six channels |
| `group_mean_abs_z`, `group_max_abs_z` | Whether channels deviate together or one is isolated |
| `group_n_channels_z_gt3` | Breadth of an excursion — one channel or the whole unit |
| `pairdiff_<a>_<b>` | Deviation between each correlated pair |
| `group_channels_present` | Data availability, so missing channels are not read as anomalies |

Per-channel columns are prefixed (`channel_41__v_mean`) and limited to `GROUP_KEEP`, so this table stays around 44 columns rather than 150. Full feature sets remain in the per-channel files.


In [8]:
def build_group_table(channel_tables, keep=GROUP_KEEP):
    indexes = [table.index for table in channel_tables.values()]
    if not all(index.equals(indexes[0]) for index in indexes[1:]):
        print("warning: channel grids differ, reindexing to the union")
    grid = indexes[0]
    for index in indexes[1:]:
        grid = grid.union(index)

    frames = []
    for channel, table in channel_tables.items():
        columns = [c for c in keep if c in table.columns]
        frames.append(table.reindex(grid)[columns].add_prefix(f"{channel}__"))
    group = pd.concat(frames, axis=1)

    levels = pd.concat(
        {channel: table["v_mean"] for channel, table in channel_tables.items()},
        axis=1,
    ).reindex(grid)
    zscores = pd.concat(
        {channel: table["robust_z_ctx"] for channel, table in channel_tables.items()},
        axis=1,
    ).reindex(grid)

    group["group_v_mean"] = levels.mean(axis=1, skipna=True)
    group["group_v_std"] = levels.std(axis=1, skipna=True)
    group["group_channels_present"] = levels.notna().sum(axis=1)
    group["group_mean_abs_z"] = zscores.abs().mean(axis=1, skipna=True)
    group["group_max_abs_z"] = zscores.abs().max(axis=1, skipna=True)
    group["group_n_channels_z_gt3"] = zscores.abs().gt(3).sum(axis=1)

    for left, right in PAIR_CANDIDATES:
        if left in channel_tables and right in channel_tables:
            group[f"pairdiff_{left}_{right}"] = (
                levels[left] - levels[right]
            )

    group.index.name = "window_start"
    return group


if len(group_inputs) == len(GROUP8_CHANNELS):
    group_table = build_group_table(group_inputs)

    # The label helper needs a group-level sample count for point annotations.
    sample_count_cols = [
        f"{channel}__n_samples"
        for _, channel in GROUP8_CHANNELS
        if f"{channel}__n_samples" in group_table.columns
    ]

    if not sample_count_cols:
        raise KeyError("No per-channel sample-count columns found in group_table")

    group_table["n_samples"] = (
        group_table[sample_count_cols].max(axis=1).fillna(0).astype("int32")
    )

    # Group labels: union across the six channels, plus how many carried each annotation.
    group_intervals = {
        category: merge_intervals(
            [
                interval
                for mission, channel in GROUP8_CHANNELS
                for interval in channel_intervals(labels_all, mission, channel, category)
            ]
        )
        for category in LABEL_CATEGORIES
    }
    group_table = attach_labels(group_table, group_intervals, PRIMARY_GROUP[0])
    group_table["split"] = assign_split(group_table.index, PRIMARY_GROUP[0])

    per_channel_flags = pd.concat(
        {
            channel: table["label_anomaly"].reindex(group_table.index)
            for channel, table in group_inputs.items()
        },
        axis=1,
    )
    group_table["n_channels_annotated"] = per_channel_flags.sum(axis=1, skipna=True)

    group_path = FEATURE_DIR / "Mission1" / "group8_windows.parquet"
    export = group_table.copy()
    for column in export.select_dtypes("float64").columns:
        export[column] = export[column].astype("float32")
    export.to_parquet(group_path, compression="snappy")

    print(f"group 8 table: {group_table.shape[0]:,} windows x {group_table.shape[1]} columns")
    display(
        group_table.groupby("split")[["label_anomaly", "label_rare_event"]]
        .sum().astype(int)
    )
elif group_inputs:
    print(
        f"group table skipped: {len(group_inputs)}/{len(GROUP8_CHANNELS)} group-8 channels "
        "present (QUICK_MODE builds a subset)"
    )


group 8 table: 736,416 windows x 67 columns


,label_anomaly,label_rare_event
split,,
test,9,2963
train,5282,2726
val,327,91


## 8. Long-format label export

The window tables already carry label columns, but a separate long table is easier to consume for event-level evaluation and for the FSM mapping: one row per `(window_start, channel, category)` where an annotation overlaps.

`Rare Event` and `Communication Gap` are included and must stay separate from `Anomaly` in any reported metric.


In [9]:
label_rows = []
for mission, channel in selected:
    table = None
    path = FEATURE_DIR / mission / f"{channel}_windows.parquet"
    if not path.exists():
        continue
    table = pd.read_parquet(path, columns=[
        "label_anomaly", "label_anomaly_frac",
        "label_rare_event", "label_rare_event_frac",
        "label_communication_gap", "label_communication_gap_frac",
        "split", "label_crosses_split",
    ])
    for category in LABEL_CATEGORIES:
        slug = category.lower().replace(" ", "_")
        mask = table[f"label_{slug}"].to_numpy(dtype=bool)
        if not mask.any():
            continue
        subset = table.loc[mask, [f"label_{slug}_frac", "split", "label_crosses_split"]].copy()
        subset = subset.rename(columns={f"label_{slug}_frac": "overlap_frac"})
        subset["Mission"] = mission
        subset["Channel"] = channel
        subset["Category"] = category
        subset["window_start"] = subset.index
        label_rows.append(subset.reset_index(drop=True))

if label_rows:
    labels_windows = pd.concat(label_rows, ignore_index=True)[
        ["Mission", "Channel", "Category", "window_start", "overlap_frac",
         "split", "label_crosses_split"]
    ]
    labels_windows.to_parquet(FEATURE_DIR / "labels_windows.parquet", index=False)
    print(f"labels_windows.parquet: {len(labels_windows):,} rows")
    display(
        labels_windows.groupby(["Mission", "Category"]).size().rename("rows").to_frame()
    )
else:
    print("no labelled windows available (is QUICK_MODE on?)")


labels_windows.parquet: 88,783 rows


rows
Mission  Category                
Mission1 Anomaly            41287
         Communication Gap   1452
         Rare Event         35658
Mission2 Anomaly              112
         Rare Event         10274

## 9. QA checks

Checks that would silently ruin a model if skipped:

| Check | Why it matters |
| --- | --- |
| Split purity | windows must not appear in two splits |
| Grid regularity | context features assume a regular grid |
| Label balance per split | a split with zero positives cannot evaluate anything |
| Boundary-spanning labels | an annotation half in train, half in test leaks the answer |
| Train-only statistics | confirms scaling data was never computed on validation/test |
| Coverage sanity | windows with almost no data must be visible, not hidden |

Expected split profiles (see section 1):

| Mission | train | val | test |
| --- | --- | --- | --- |
| Mission 1 | 2000 → 2010 | 2010 → 2012 | 2012 → 2013 |
| Mission 2 | 2000 → mid-2001 | mid-2001 → mid-2002 | mid-2002 → mid-2003 |

Group 8 carries annotations in all three periods. `channel_66` has none in 2012–2013, so it shows `test_pos = 0` — expected for a baseline-only stream, and the reason its baseline rows are marked `evaluable = False` rather than scored as zero.


In [10]:
qa_rows = []
for mission, channel in selected:
    path = FEATURE_DIR / mission / f"{channel}_windows.parquet"
    if not path.exists():
        continue
    table = pd.read_parquet(path)

    split_counts = table["split"].value_counts()
    boundary = table.index.max() - table.index.min()
    expected_rows = int(round(boundary / pd.Timedelta(minutes=WINDOW_MINUTES)))
    profile = SPLIT_PROFILES[mission]
    qa_rows.append(
        {
            "Channel": channel,
            "Mission": mission,
            "rows": len(table),
            "grid_ok": abs(len(table) - expected_rows) <= 1,
            "unassigned": int(split_counts.get("unassigned", 0)),
            "train": int(split_counts.get("train", 0)),
            "val": int(split_counts.get("val", 0)),
            "test": int(split_counts.get("test", 0)),
            "train_pos": int(table.loc[table["split"].eq("train"), "label_anomaly"].sum()),
            "val_pos": int(table.loc[table["split"].eq("val"), "label_anomaly"].sum()),
            "test_pos": int(table.loc[table["split"].eq("test"), "label_anomaly"].sum()),
            "crosses_split": int(table["label_crosses_split"].sum()),
            "coverage_lt_half": int(table["coverage"].lt(0.5).sum()),
            "test_window": f"{profile['test'][0]} -> {profile['test'][1]}",
        }
    )

qa = pd.DataFrame(qa_rows)
display(qa)

assert qa["unassigned"].sum() == 0, "some windows fall outside the configured split profiles"
assert qa["grid_ok"].all(), "window grid is not regular"

no_test = qa.loc[qa["test_pos"].eq(0), "Channel"].tolist()
no_val = qa.loc[qa["val_pos"].eq(0), "Channel"].tolist()
print("no train positives :", qa.loc[qa["train_pos"].eq(0), "Channel"].tolist() or "none")
print("no val positives   :", no_val or "none")
print("no test positives  :", no_test or "none")
print("(streams without test positives cannot be evaluated on this split; channel_66 is baseline-only by design)")
print("boundary-spanning windows excluded from scoring:", int(qa["crosses_split"].sum()))


,Channel,Mission,rows,grid_ok,unassigned,train,val,test,train_pos,val_pos,test_pos,crosses_split,coverage_lt_half,test_window
0,channel_41,Mission1,736416,True,0,526032,105120,105264,5233,325,9,0,5349,2012-01-01 -> 2014-01-01
1,channel_42,Mission1,736416,True,0,526032,105120,105264,5235,327,8,0,5349,2012-01-01 -> 2014-01-01
2,channel_43,Mission1,736416,True,0,526032,105120,105264,5261,327,8,0,5349,2012-01-01 -> 2014-01-01
3,channel_44,Mission1,736416,True,0,526032,105120,105264,5259,325,8,0,5349,2012-01-01 -> 2014-01-01
4,channel_45,Mission1,736416,True,0,526032,105120,105264,5228,326,8,0,5349,2012-01-01 -> 2014-01-01
5,channel_46,Mission1,736416,True,0,526032,105120,105264,5243,323,8,0,5349,2012-01-01 -> 2014-01-01
6,channel_13,Mission2,183888,True,0,74448,52560,56880,57,9,0,0,3,2002-06-01 -> 2004-01-01
7,channel_20,Mission2,183888,True,0,74448,52560,56880,42,4,0,0,3,2002-06-01 -> 2004-01-01
8,channel_66,Mission1,736416,True,0,526032,105120,105264,7065,761,0,0,799,2012-01-01 -> 2014-01-01


no train positives : none
no val positives   : none
no test positives  : ['channel_13', 'channel_20', 'channel_66']
(streams without test positives cannot be evaluated on this split; channel_66 is baseline-only by design)
boundary-spanning windows excluded from scoring: 0


## 10. Baseline reference detection

Not a model — a yardstick. Three trivial rules are scored on the **test split**, at the same event level the real model will be judged on:

| Rule | Score | Score on |
| --- | --- | --- |
| Zero rule | `zero_frac` | windows |
| Robust-z rule | `abs(robust_z_ctx)` | windows |
| Step rule | `abs(mean_delta_w)` | windows |

The threshold for each rule is the **train nominal p99.5**, so no test information is used. Predicted runs of consecutive windows are merged into events, and an event counts as a true positive if it overlaps a ground-truth `Anomaly` interval.

Two things this section must show:

1. every rule fires at roughly the expected rate on nominal data (sanity), and
2. how weak trivial rules really are, so the model has to beat a concrete number.

Windows with `label_crosses_split` are dropped from scoring. Events are matched one-to-one, so a single long detour cannot collect several true positives.

Rows with `evaluable = False` have **no** ground-truth `Anomaly` events in the test split (for example `channel_66`, whose annotations stop in 2011). Their precision/recall are `NaN`, not zero — an absent ground truth is not a perfect detector.


In [11]:
def runs_to_intervals(flag, index, window_minutes=WINDOW_MINUTES):
    idx = np.flatnonzero(np.asarray(flag, dtype=bool))
    if idx.size == 0:
        return []
    breaks = np.flatnonzero(np.diff(idx) > 1)
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends = np.r_[idx[breaks], idx[-1]]
    step = pd.Timedelta(minutes=window_minutes)
    return [(index[s], index[e] + step) for s, e in zip(starts, ends)]


def match_events(predicted, ground_truth):
    used = set()
    true_positives = 0
    for pred_start, pred_end in predicted:
        for position, (gt_start, gt_end) in enumerate(ground_truth):
            if position in used:
                continue
            if pred_start <= gt_end and gt_start <= pred_end:
                used.add(position)
                true_positives += 1
                break
    return true_positives, len(predicted) - true_positives, len(ground_truth) - len(used)


def score_rule(table, score_column, threshold, split="test"):
    split_table = table[table["split"].eq(split) & ~table["label_crosses_split"]]
    if split_table.empty:
        return None
    predicted = runs_to_intervals(
        split_table[score_column].fillna(0).gt(threshold).to_numpy(),
        split_table.index,
    )
    ground_truth = runs_to_intervals(
        split_table["label_anomaly"].to_numpy(), split_table.index
    )
    tp, fp, fn = match_events(predicted, ground_truth)
    evaluable = len(ground_truth) > 0
    precision = tp / len(predicted) if predicted else 0.0
    recall = tp / len(ground_truth) if ground_truth else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    nominal_hours = (
        split_table["coverage"].fillna(0).sum() * WINDOW_MINUTES / 60.0
    )
    return {
        "Channel": table.attrs.get("channel", ""),
        "rule": score_column,
        "threshold": threshold,
        "evaluable": evaluable,
        "gt_events": len(ground_truth),
        "pred_events": len(predicted),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": round(precision, 3) if evaluable else np.nan,
        "recall": round(recall, 3) if evaluable else np.nan,
        "f1": round(f1, 3) if evaluable else np.nan,
        "fp_per_hour": round(fp / nominal_hours, 4) if nominal_hours else np.nan,
    }


baseline_rows = []
for mission, channel in selected:
    path = FEATURE_DIR / mission / f"{channel}_windows.parquet"
    if not path.exists():
        continue
    table = pd.read_parquet(path)
    table.attrs["channel"] = channel

    train_nominal = (
        table["split"].eq("train")
        & ~table["label_anomaly"]
        & ~table["label_rare_event"]
    )
    for score_column in ("zero_frac", "robust_z_ctx", "mean_delta_w"):
        if score_column not in table:
            continue
        scores = table.loc[train_nominal, score_column].abs().dropna()
        if scores.empty:
            continue
        threshold = float(scores.quantile(0.995))
        row = score_rule(table, score_column, threshold)
        if row:
            baseline_rows.append(row)

baseline = pd.DataFrame(baseline_rows)
display(baseline.sort_values(["Channel", "f1"], ascending=[True, False]))


,Channel,rule,threshold,evaluable,gt_events,pred_events,tp,fp,fn,precision,recall,f1,fp_per_hour
18,channel_13,zero_frac,0.000000,False,0,0,0,0,0,NaN,NaN,NaN,0.0000
19,channel_13,robust_z_ctx,12.790986,False,0,91,0,91,0,NaN,NaN,NaN,0.0096
20,channel_13,mean_delta_w,0.007184,False,0,46,0,46,0,NaN,NaN,NaN,0.0049
21,channel_20,zero_frac,0.000000,False,0,0,0,0,0,NaN,NaN,NaN,0.0000
22,channel_20,robust_z_ctx,10.709079,False,0,105,0,105,0,NaN,NaN,NaN,0.0111
23,channel_20,mean_delta_w,0.000396,False,0,81,0,81,0,NaN,NaN,NaN,0.0085
1,channel_41,robust_z_ctx,2.413184,True,9,58,3,55,6,0.052,0.333,0.090,0.0031
2,channel_41,mean_delta_w,0.008223,True,9,103,5,98,4,0.049,0.556,0.089,0.0056
0,channel_41,zero_frac,0.000000,True,9,0,0,0,9,0.000,0.000,0.000,0.0000
5,channel_42,mean_delta_w,0.011666,True,8,94,4,90,4,0.043,0.500,0.078,0.0051


## 11. Export the handoff package

The manifest is the contract with the model repository. It records schema version, window length, per-channel cadence, split boundaries, the full feature list, row and label counts, and the train-only statistics used for scaling.

`feature_tables.zip` is what you download from Colab. Keep it out of the repository — the model repo should read Parquet, not committed binaries.


In [12]:
from datetime import datetime, timezone
import shutil

manifest = {
    "schema_version": SCHEMA_VERSION,
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "window_minutes": WINDOW_MINUTES,
    "label_categories": LABEL_CATEGORIES,
    "positive_category": POSITIVE_CATEGORY,
    "features": sorted({f for entry in summaries for f in entry["features"]}),
    "splits": {
        mission: {name: list(bounds) for name, bounds in profile.items()}
        for mission, profile in SPLIT_PROFILES.items()
    },
    "primary_group": {"mission": PRIMARY_GROUP[0], "group": PRIMARY_GROUP[1],
                      "channels": [c for _, c in GROUP8_CHANNELS]},
    "channels": summaries,
    "notes": [
        "Channels are anonymized; no physical sensor identity is claimed.",
        "Target=YES is an anomaly-benchmark flag, not a sensor type.",
        "Rare Event and Communication Gap are separate from Anomaly.",
        "All context features are causal (backward-looking only).",
        "train_stats.json is computed on train-split nominal windows only.",
        "Generated/mock data and Raspberry Pi readings are excluded.",
    ],
}

with open(FEATURE_DIR / "manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

train_stats = {entry["Channel"]: entry["train_stats"] for entry in summaries}
with open(FEATURE_DIR / "train_stats.json", "w", encoding="utf-8") as handle:
    json.dump(train_stats, handle, indent=2)

print(f"features per channel: {len(manifest['features'])}")
print("channels:", ", ".join(entry["Channel"] for entry in summaries))

archive_base = WORK_DIR / "feature_tables"
if archive_base.with_suffix(".zip").exists():
    archive_base.with_suffix(".zip").unlink()
shutil.make_archive(str(archive_base), "zip", FEATURE_DIR)

for path in sorted(FEATURE_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(FEATURE_DIR)}  {path.stat().st_size / 1024**2:.1f} MiB")

print(f"\nzip: {archive_base.with_suffix('.zip')}")


features per channel: 30
channels: channel_41, channel_42, channel_43, channel_44, channel_45, channel_46, channel_13, channel_20, channel_66
Mission1/channel_41_windows.parquet  34.7 MiB
Mission1/channel_42_windows.parquet  36.0 MiB
Mission1/channel_43_windows.parquet  31.6 MiB
Mission1/channel_44_windows.parquet  35.8 MiB
Mission1/channel_45_windows.parquet  34.9 MiB
Mission1/channel_46_windows.parquet  36.0 MiB
Mission1/channel_66_windows.parquet  10.6 MiB
Mission1/group8_windows.parquet  86.9 MiB
Mission2/channel_13_windows.parquet  13.0 MiB
Mission2/channel_20_windows.parquet  13.9 MiB
labels_windows.parquet  0.3 MiB
manifest.json  0.1 MiB
train_stats.json  0.0 MiB

zip: /content/feature_engineering/feature_tables.zip


## 12. Handoff notes for the model developer

### What you are receiving

- **Group 8** (`channel_41`–`channel_46`): six per-channel tables plus `group8_windows.parquet`. Primary benchmark.
- **`channel_66`**: the deliberately easy baseline stream. Use it to prove the pipeline detects *anything*; do not use it to claim detection quality.
- **Mission 2 `channel_13`, `channel_20`**: separate comparison streams. Do **not** concatenate them with group 8 — different mission, cadence, range, and label vocabulary.
- `labels_windows.parquet` and `manifest.json` for event-level evaluation and provenance.

### What the model must do

1. Train on `split == "train"` only, select on `val`, report on `test`. Never refit scalers on `val`/`test`.
2. Use `train_stats.json` for standardisation, or recompute from train rows only.
3. Score at **event level**: merge consecutive positive windows into events and match one-to-one against `Anomaly` windows. Sample-level accuracy is meaningless at this class balance.
4. Beat the three baselines in section 10. At minimum, beat the zero rule.
5. Report `Anomaly` separately from `Rare Event` and `Communication Gap`.

### Traps to avoid

- **Zero-related leakage.** `channel_66` and `channel_41` both contain zero-valued anomalies. A model that only learns "value == 0" will look impressive on those two and fail elsewhere. Check performance with zero-dominated windows removed.
- **Rare Event inflation.** Mission 2 streams have ~600 labelled windows each but only 11–12 are `Anomaly`. Counting all labels as positives would misrepresent the task.
- **Duplicate events across channels.** Group 8 labels repeat per channel. Use the group union (98 `Anomaly`) for group-level claims.
- **Sparse windows.** `coverage < 0.5` means the window is mostly missing data. Exclude it or model it explicitly rather than treating it as calm.
- **Boundary-spanning annotations.** Drop rows where `label_crosses_split` is true before scoring.
- **Missing windows ≠ normal windows.** `is_present` and `coverage` separate "quiet" from "not observed".

### Limits of this dataset

These are **anonymized ESA telemetry channels**, not BMP280, INA219, or IMU data. A model trained here is a generic telemetry anomaly detector; success does not automatically transfer to the generated sensor-shaped stream. Validate transfer separately, and keep the generated stream out of training.

Native cadences are 18 s and 30 s. Do not upsample to 10 Hz and describe it as native sensor sampling; the 10 records/s requirement applies to replay throughput.
